# US Election 2020 Tweets — Data Analysis Report

- **Prepared by:** Decsi (2506626) & Imanakhunova (2508355)
- **Dataset:** US Election 2020 Tweets

In [ ]:
%run ./Includes/Classroom-Setup

## 1. Data Ingestion & Exploration

We use `dbutils.fs.ls()` to list the tweet dataset files on the Unity Catalog volume and inspect their sizes.

In [ ]:
%fs ls /Volumes/dbx_course/source/files/assignment/us_election_2020_tweets/

Preview the first few lines of the Trump tweets CSV to understand the column structure before loading.

In [ ]:
%fs head /Volumes/dbx_course/source/files/assignment/us_election_2020_tweets/hashtag_donaldtrump.csv

Load both CSV files using `spark.read.csv()` with `header=True` and `inferSchema=True`. We also set `multiLine=True` and `escape='"'` since tweet text may contain newlines and quoted strings.

In [ ]:
trump_auto = spark.read.csv(
    "/Volumes/dbx_course/source/files/assignment/us_election_2020_tweets/hashtag_donaldtrump.csv",
    header=True, inferSchema=True, multiLine=True, escape='"'
)
trump_auto.printSchema()
display(trump_auto.limit(5))

biden_auto = spark.read.csv(
    "/Volumes/dbx_course/source/files/assignment/us_election_2020_tweets/hashtag_joebiden.csv",
    header=True, inferSchema=True, multiLine=True, escape='"'
)
biden_auto.printSchema()
display(biden_auto.limit(5))

Define the schema manually using a DDL-formatted string. This gives explicit control over column names and data types, and avoids the extra pass Spark needs for `inferSchema`.

In [ ]:
tweets_schema = """
`created_at` STRING,
`tweet_id` STRING,
`tweet` STRING,
`likes` DOUBLE,
`retweet_count` DOUBLE,
`source` STRING,
`user_id` STRING,
`user_name` STRING,
`user_screen_name` STRING,
`user_description` STRING,
`user_join_date` STRING,
`user_followers_count` DOUBLE,
`user_location` STRING,
`lat` DOUBLE,
`long` DOUBLE,
`city` STRING,
`country` STRING,
`continent` STRING,
`state` STRING,
`state_code` STRING,
`collected_at` STRING
"""

Load both datasets using the manually defined schema. Using `.schema()` applies our DDL string directly.

In [ ]:
df_trump = (
    spark.read
         .schema(tweets_schema)
         .option("header", True)
         .option("multiLine", True)
         .option("escape", '"')
         .csv("/Volumes/dbx_course/source/files/assignment/us_election_2020_tweets/hashtag_donaldtrump.csv")
)

df_biden = (
    spark.read
         .schema(tweets_schema)
         .option("header", True)
         .option("multiLine", True)
         .option("escape", '"')
         .csv("/Volumes/dbx_course/source/files/assignment/us_election_2020_tweets/hashtag_joebiden.csv")
)

Show the total number of rows using `.count()` and display a sample from each dataset.

In [ ]:
print(f"Trump tweets: {df_trump.count():,}")
display(df_trump.limit(5))

print(f"Biden tweets: {df_biden.count():,}")
display(df_biden.limit(5))

## 2. SQL Queries

Register both DataFrames as temporary views so we can query them with SQL. Then read them back through `spark.table()` to confirm they are accessible.

In [ ]:
df_trump.createOrReplaceTempView("trump_tweets")
df_trump_sql = spark.table("trump_tweets")

Register the Biden DataFrame as a temp view as well.

In [ ]:
df_biden.createOrReplaceTempView("biden_tweets")
df_biden_sql = spark.table("biden_tweets")

**SQL Query 1 — Filtering:** Find the most-liked Trump tweets with more than 10,000 likes. This surfaces the most viral content in the dataset.

In [ ]:
%sql
SELECT tweet, likes, retweet_count
FROM trump_tweets
WHERE likes > 10000
ORDER BY likes DESC
LIMIT 5

Same query for Biden tweets — lets us compare engagement levels across the two hashtags.

In [ ]:
%sql
SELECT tweet, likes, retweet_count
FROM biden_tweets
WHERE likes > 10000
ORDER BY likes DESC
LIMIT 5

**SQL Query 2 — Aggregation:** Calculate average likes and retweets grouped by date for Trump tweets. This reveals daily engagement trends during the election period.

In [ ]:
%sql
SELECT created_at,
       AVG(likes) AS avg_likes,
       AVG(retweet_count) AS avg_retweets
FROM trump_tweets
GROUP BY created_at
ORDER BY avg_likes DESC
LIMIT 10

## 3. DataFrame Transformations

**Filter 1** — Identify influential content by selecting tweets with more than 1,000 retweets (viral content) posted by users with at least 100K followers (authoritative voices).

In [ ]:
from pyspark.sql.functions import col

high_engagement_trump = df_trump.filter(
    (col("retweet_count") > 1000) &
    (col("user_followers_count") >= 100000)
)

print(f"High-engagement influencer Trump tweets: {high_engagement_trump.count():,}")
display(high_engagement_trump.select(
    "user_screen_name", "user_followers_count",
    "retweet_count", "likes", "created_at", "tweet"
).limit(5))

**Filter 2** — Focus on the critical final month before the election: October 2020 Biden tweets from the United States. This isolates domestic activity during the campaign's most intense period.

In [ ]:
october_tweets = df_biden.filter(
    (col("created_at") >= "2020-10-01") &
    (col("created_at") < "2020-11-01") &
    (col("country") == "United States of America")
)

print(f"Biden tweets from October 2020 in USA: {october_tweets.count():,}")
display(october_tweets.select(
    "created_at", "user_screen_name", "user_location", "tweet"
).limit(5))

Use `select()` to pick relevant columns and `withColumn()` to create three computed columns:
- **engagement_ratio**: likes per retweet — measures content quality
- **tweet_length_category**: short / medium / long based on character count
- **is_reply**: heuristic for whether the tweet mentions another user

In [ ]:
from pyspark.sql.functions import col, when, length

transformed_df = df_trump.select(
    col("tweet_id"),
    col("user_screen_name"),
    col("created_at"),
    col("retweet_count"),
    col("likes"),
    col("tweet"),
    col("source")
).withColumn(
    "engagement_ratio",
    when(col("retweet_count") > 0, col("likes") / col("retweet_count")).otherwise(0)
).withColumn(
    "tweet_length_category",
    when(length(col("tweet")) <= 100, "Short")
    .when(length(col("tweet")) <= 200, "Medium")
    .otherwise("Long")
).withColumn(
    "is_reply",
    col("tweet").contains("@")
)

display(transformed_df.limit(5))

Sort by engagement quality (likes per retweet) descending, then by retweet count as a tiebreaker. We filter for tweets with at least 100 retweets to ensure the ratio is meaningful, and limit to the top 5.

In [ ]:
top_quality_tweets = transformed_df.filter(
    col("retweet_count") > 100
).sort(
    col("engagement_ratio").desc(),
    col("retweet_count").desc()
).limit(5)

print("Top 5 tweets by engagement quality:")
display(top_quality_tweets.select(
    "user_screen_name", "engagement_ratio",
    "retweet_count", "likes", "tweet_length_category"
))

Check for duplicate tweets. Duplicates in tweet data might indicate retweets, bot activity, or data ingestion issues. We compare total rows vs. distinct `tweet_id` values and distinct tweet contents.

In [ ]:
total_tweets = df_trump.count()
distinct_tweets = df_trump.select("tweet_id").distinct().count()
duplicate_count = total_tweets - distinct_tweets

print(f"Total Trump tweets: {total_tweets:,}")
print(f"Distinct tweet IDs: {distinct_tweets:,}")
print(f"Duplicate count: {duplicate_count:,}")

distinct_content = df_trump.select("tweet").distinct().count()
print(f"Distinct tweet contents: {distinct_content:,}")

if duplicate_count > 0:
    duplicates = df_trump.groupBy("tweet_id").count().filter(col("count") > 1)
    print("\nSample duplicate tweet IDs:")
    display(duplicates.limit(5))
else:
    print("\nNo duplicate tweet IDs found — data quality is good")

**Duplicate Analysis:**

- **tweet_id duplicates** would indicate data ingestion errors or system retries
- **Content duplicates** likely represent retweets or quote tweets with the same text (Twitter assigns unique IDs to retweets)
- The `dropDuplicates()` operation removes exact row matches; since `tweet_id` is the primary key, we expect few or no exact duplicates

Apply `dropDuplicates()` to remove any exact duplicate rows from both datasets. We verify by comparing counts before and after.

In [ ]:
df_trump_clean = df_trump.dropDuplicates()
df_biden_clean = df_biden.dropDuplicates()

print(f"Trump tweets after deduplication: {df_trump_clean.count():,}")
print(f"Biden tweets after deduplication: {df_biden_clean.count():,}")

if df_trump_clean.count() == df_trump.count():
    print("No exact duplicates found in Trump dataset")
if df_biden_clean.count() == df_biden.count():
    print("No exact duplicates found in Biden dataset")

## 4. Aggregations

**Built-in aggregation method 1 — `.count()`:** Count tweets by country to see the geographic distribution of Trump-hashtag activity.

In [ ]:
country_counts = (
    df_trump_clean
    .filter(col("country").isNotNull())
    .groupBy("country")
    .count()
    .orderBy(col("count").desc())
)

print("Trump tweets by country:")
display(country_counts.limit(10))

**Built-in aggregation method 2 — `.avg()`:** Calculate average likes per tweet source platform. This reveals which platforms produce the highest-engagement content.

In [ ]:
source_avg_likes = (
    df_trump_clean
    .filter(col("source").isNotNull())
    .groupBy("source")
    .avg("likes")
    .orderBy(col("avg(likes)").desc())
)

print("Average likes by source platform:")
display(source_avg_likes.limit(10))

**Complex aggregation with `groupBy().agg()`:** Analyze how tweet length affects engagement. We use three self-learned aggregate functions — `stddev()` (standard deviation of retweets), `countDistinct()` (unique authors per category), and `percentile_approx()` (median likes) — alongside `count()` and `avg()` from class.

In [ ]:
from pyspark.sql.functions import (
    stddev, countDistinct, percentile_approx,
    col, count, avg, length, when
)

length_engagement = (
    df_trump_clean
    .filter(col("tweet").isNotNull())
    .withColumn("tweet_length", length(col("tweet")))
    .withColumn("length_category",
        when(col("tweet_length") <= 50, "Very Short")
        .when(col("tweet_length") <= 100, "Short")
        .when(col("tweet_length") <= 200, "Medium")
        .otherwise("Long")
    )
    .groupBy("length_category")
    .agg(
        count("*").alias("tweet_count"),
        avg("likes").alias("avg_likes"),
        stddev("retweet_count").alias("retweet_stddev"),
        countDistinct("user_id").alias("unique_authors"),
        percentile_approx("likes", 0.5).alias("median_likes")
    )
    .orderBy(col("tweet_count").desc())
)

print("Tweet Length vs Engagement Analysis:")
display(length_engagement)

## 5. Visualizations

**Visualization 1 — Bar Chart:** Tweet volume and average engagement by tweet-length category. After running `display()`, click the chart icon (+) in the output toolbar and select **Bar Chart** with `length_cat` on the x-axis and `tweet_count` / `avg_likes` on the y-axis.

In [ ]:
from pyspark.sql.functions import col, count, avg, length, when

length_analysis = (
    df_trump_clean
    .filter(col("tweet").isNotNull())
    .withColumn("length_cat",
        when(length(col("tweet")) <= 50, "Very Short (0-50)")
        .when(length(col("tweet")) <= 100, "Short (51-100)")
        .when(length(col("tweet")) <= 200, "Medium (101-200)")
        .otherwise("Long (200+)")
    )
    .groupBy("length_cat")
    .agg(
        count("*").alias("tweet_count"),
        avg("retweet_count").alias("avg_retweets"),
        avg("likes").alias("avg_likes")
    )
    .orderBy(col("tweet_count").desc())
)

display(length_analysis)

**Visualization 2 — Line Chart:** Compare Trump vs Biden tweet volumes by country (top 10). After running `display()`, click the chart icon and select **Line Chart** with `country` on the x-axis, `count` on the y-axis, and group by `candidate`.

In [ ]:
from pyspark.sql.functions import lit

country_comparison = (
    df_trump_clean.filter(col("country").isNotNull())
    .groupBy("country").count()
    .withColumn("candidate", lit("Trump"))
    .union(
        df_biden_clean.filter(col("country").isNotNull())
        .groupBy("country").count()
        .withColumn("candidate", lit("Biden"))
    )
)

top_countries = (
    country_comparison.groupBy("country")
    .sum("count")
    .withColumnRenamed("sum(count)", "total")
    .orderBy(col("total").desc())
    .limit(10)
    .select("country")
)

country_viz = (
    country_comparison
    .join(top_countries, "country")
    .orderBy(col("count").desc())
)

display(country_viz)

## 6. Delta Lake

Write the cleaned Trump tweets DataFrame to **Delta format** under the working directory provided by Classroom-Setup (`DA.paths.workdir`).

In [ ]:
delta_path = f"{DA.paths.workdir}/trump_tweets_delta"
df_trump_clean.write.format("delta").mode("overwrite").save(delta_path)

List the Delta folder contents and confirm that the `_delta_log/` directory exists alongside the data files.

In [ ]:
display(dbutils.fs.ls(delta_path))

**What is `_delta_log`?**

The `_delta_log` directory is the **transaction log** of a Delta Lake table. It stores an ordered record of every transaction (write, update, delete) as JSON files. This enables Delta Lake's key features:

- **ACID transactions** — each commit is atomic and recorded in the log, preventing partial or corrupt writes
- **Time travel** — the full history of changes allows querying previous versions of the data
- **Schema enforcement** — the log tracks the table schema and rejects incompatible writes
- **Audit history** — every change is traceable through the commit log

Register the Delta data as a **managed table** in the catalog using `.saveAsTable()`, then query it with SQL to verify the data is accessible as a table.

In [ ]:
df_trump_clean.write.format("delta").mode("overwrite").saveAsTable("trump_tweets_delta")

Query the newly registered Delta table using SQL to verify the data is accessible through the catalog.

In [ ]:
%sql
SELECT source,
       COUNT(*) AS tweet_count,
       ROUND(AVG(likes), 2) AS avg_likes
FROM trump_tweets_delta
GROUP BY source
ORDER BY tweet_count DESC
LIMIT 10

## 7. Additional pyspark.sql.functions

We apply two functions not used elsewhere in this notebook:

- **`upper()`** — normalizes tweet source names to uppercase for consistent comparison
- **`to_date()`** — extracts the date portion from the string timestamp, enabling proper date-based grouping and trend analysis

In [ ]:
from pyspark.sql.functions import upper, to_date, substring

df_extra = (
    df_trump_clean
    .withColumn("source_upper", upper(col("source")))
    .withColumn("tweet_date",
        to_date(substring(col("created_at"), 1, 10), "yyyy-MM-dd")
    )
    .select("tweet_id", "source", "source_upper",
            "created_at", "tweet_date", "likes", "retweet_count")
)

display(df_extra.limit(10))

## 8. Null Handling with DataFrameNaFunctions

The location-related columns (`country`, `city`, `state`, `lat`, `long`) likely contain many nulls because not all tweets are geotagged. We use **`.na.fill()`** to replace nulls in `country`, `city`, and `state` with `"Unknown"` so that aggregations include these records rather than silently dropping them. For the numeric coordinates `lat` and `long`, we use **`.na.drop()`** to remove rows where both are null, since filling them with arbitrary values would be misleading for geographic analysis.

In [ ]:
before_count = df_trump_clean.count()
null_country = df_trump_clean.filter(col("country").isNull()).count()
null_city = df_trump_clean.filter(col("city").isNull()).count()
null_lat = df_trump_clean.filter(col("lat").isNull()).count()

print(f"Total rows: {before_count:,}")
print(f"Null country: {null_country:,} ({100*null_country/before_count:.1f}%)")
print(f"Null city: {null_city:,} ({100*null_city/before_count:.1f}%)")
print(f"Null lat/long: {null_lat:,} ({100*null_lat/before_count:.1f}%)")

df_filled = df_trump_clean.na.fill(
    {"country": "Unknown", "city": "Unknown", "state": "Unknown"}
)
df_geo = df_trump_clean.na.drop(subset=["lat", "long"])

filled_nulls = df_filled.filter(col("country").isNull()).count()
print(f"\nAfter na.fill - null countries: {filled_nulls}")
print(f"After na.drop on lat/long - remaining rows: {df_geo.count():,}")

## 10. Data Story

### Geographic and Platform Patterns in Election Tweets

Our analysis of Trump-related tweets during the 2020 US election reveals striking geographic and platform-usage patterns. The vast majority of tweets originate from the United States, but there is notable international engagement — particularly from countries like India, the United Kingdom, and Canada. This suggests that the US presidential election is not merely a domestic event but a global conversation. The engagement metrics (likes and retweets) vary significantly across countries, with US-based tweets generally receiving higher engagement, likely due to the larger domestic audience and higher personal stakes.

The source-platform analysis shows a clear dominance of mobile devices (Twitter for iPhone and Twitter for Android) over desktop clients. This aligns with the real-time, reactive nature of political discourse on Twitter — people tweet about political events as they happen, often from their phones. Interestingly, the maximum engagement metrics differ substantially across platforms, with some web-based clients showing surprisingly high peak engagement, possibly indicating institutional or media accounts that tend to use desktop clients.

### Tweet Content and Engagement Dynamics

The tweet-length vs. engagement analysis reveals a counterintuitive finding: shorter tweets do not necessarily perform worse than longer ones. While medium-length tweets (100–200 characters) show the highest average engagement, very short tweets also demonstrate strong performance — suggesting that concise, punchy messages can be just as impactful as longer, more detailed ones. The high standard deviation in retweet counts across all length categories indicates that viral content is not determined by length alone but likely depends more on the author's influence, timing, and the tweet's emotional resonance.

The presence of significant null values in geographic fields (country, city, lat/long) is also worth noting — a large portion of users either disable location sharing or do not provide geographic metadata, which limits the completeness of any geographic analysis. Future investigation could focus on sentiment analysis of tweet content to understand whether positive or negative framing drives more engagement, and whether this pattern differs between Trump and Biden-related hashtags.